<a href="https://colab.research.google.com/github/w3aarush/deep-learning/blob/main/EfficientNet_multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %pip install tensorflow

In [2]:
import tensorflow as tf
# from tensorflow.keras.layers.experimental import preprocessing
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Attention
# from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
import matplotlib.pyplot as plt

In [3]:
# %pip install opencv-python

In [4]:
# %pip install --extra-index-url=https://pypi.nvidia.com cuml-cu13

In [5]:
from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2S, preprocess_input
from google.colab.patches import cv2_imshow
import pandas as pd
import numpy as np
import seaborn as sns
# import imutils
import time
import cv2
from cuml import SVC # for python 3.11
# from sklearn.svm import SVC

In [6]:
# Install Kaggle API
!pip install -q kaggle

# Upload kaggle.json
from google.colab import files
files.upload()

# Setup Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Download DATASET (not competition)
!kaggle datasets download -d mariaherrerot/aptos2019

# Unzip
!unzip -q aptos2019.zip -d aptos2019

# Check files
!ls aptos2019

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/mariaherrerot/aptos2019
License(s): unknown
100% 8.01G/8.01G [00:49<00:00, 173MB/s]

test.csv  test_images  train_1.csv  train_images  valid.csv  val_images


In [7]:
base_dir = '/content/aptos2019'
train_dir = '/content/aptos2019/train_images/train_images/'
validation_dir = '/content/aptos2019/val_images/val_images/'
test_dir = '/content/aptos2019/test_images/test_images/'

In [8]:
import os

In [9]:
print(os.listdir(train_dir))
print(os.listdir(validation_dir))
print(os.listdir(test_dir))

['721214151233.png', 'add1d681d712.png', '878e356c8fc9.png', '2f4e81787d9b.png', '87295c5fa1cc.png', 'bf7b4eae7ad0.png', '44e0d56e9d42.png', '4ccee4db09b6.png', '286e9981dd9b.png', '6298468d7d75.png', 'ae57c8630249.png', '99ecdb41d5e7.png', 'cb0cc98d7e35.png', '3218a6d8eb2c.png', '38e0e28d35d3.png', '2923971566fe.png', '524f240e0c90.png', 'e0d229db881a.png', '7e4019ac7f5a.png', '683023cda6a5.png', '4fa26d065ad3.png', 'a95858e052d6.png', 'a45d77edf8d9.png', 'a8e88d4891c4.png', '89d9c071a56f.png', '70d657f8f503.png', 'd10ef306996b.png', 'c8905b8d5cf1.png', 'aad0c0ee9268.png', '94ef1d14597f.png', '2209daf71aab.png', '63363410389a.png', '9bf060db8376.png', '842d697884f6.png', '40140a925c43.png', 'dc6fa1b38b83.png', 'e32dc722eca5.png', 'e246cd89e1cc.png', '9fefe2b44795.png', '252305189b3a.png', 'b67ae80f7eba.png', '7828dd083cdc.png', 'd436c06f0490.png', '384db24ebbd7.png', 'af3b0115aad1.png', '1cb6961d141c.png', 'a19ecd0a706e.png', '6b91e99c9408.png', 'cd563556cb57.png', '8e6df9eedcd8.png',

In [10]:
NUM_CLASSES = 5
epochs = 20

In [11]:
BATCH_SIZE = 32

In [12]:
img_augmentation = Sequential(
    [
        tf.keras.layers.RandomRotation(factor=(-0.15, 0.15)),
        tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
        tf.keras.layers.RandomFlip(),
        tf.keras.layers.RandomContrast(factor=0.1),
    ],
    name="img_augmentation",
)

In [13]:
def build_EfficientNetV2S_model_SVM_without_attention(num_classes):
    IMG_SIZE = (224, 224)
    # Load the EfficientNetV2S base model without the top (classification) layer
    base_model = EfficientNetV2S(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')

    # Freeze the base model layers
    base_model.trainable = False

    # Create a new model on top of the base model
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = img_augmentation(inputs)
    x = preprocess_input(x) # EfficientNetV2S expects inputs in the [-1, 1] range after preprocessing
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    # outputs = layers.Dense(num_classes, activation='softmax')(x)
    outputs = layers.Dense(5, activation='softmax')(x)
    model = keras.Model(inputs, outputs)

    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

This code defines an image augmentation pipeline using Keras's Sequential model. It applies a series of random transformations to input images:

RandomRotation(factor=(-0.15, 0.15)): Randomly rotates images by an angle within the range of -15% to +15% of 2π radians.
RandomTranslation(height_factor=0.1, width_factor=0.1): Randomly shifts images horizontally and vertically by up to 10% of their width and height, respectively.
RandomFlip(): Randomly flips images horizontally (left-right).
RandomContrast(factor=0.1): Randomly adjusts the contrast of images by a factor within the range of [1 - 0.1, 1 + 0.1] (i.e., [0.9, 1.1]).
This img_augmentation pipeline is typically used during model training to artificially increase the size and diversity of the training dataset, helping to improve the model's generalization capabilities and reduce overfitting.

In [14]:
def unfreeze(model):
    # unfreeze the top 10 layers while leaving BatchNorm layers frozen for fine-tuning
    for layer in model.layers[-10:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    '''
    This code defines a function called unfreeze that takes a Keras model as input. Its primary purpose is to prepare a pre-trained model for fine-tuning.

Here's a breakdown:

Unfreezing Top Layers: The for loop iterates through the last 10 layers of the model (model.layers[-10:]). For each of these layers, if it's not a BatchNormalization layer, its trainable attribute is set to True. This means these layers' weights will be updated during subsequent training, allowing for fine-tuning. Batch Normalization layers are often kept frozen during fine-tuning to prevent issues with small batch sizes or drastic changes in statistics.

Optimizer Setup: It initializes an Adam optimizer with a learning rate of 1e-4 (0.0001). Adam is a popular optimization algorithm known for its efficiency.

Model Compilation: Finally, the model is compiled with the specified optimizer, loss='categorical_crossentropy', and metrics=['accuracy']. This prepares the model for training by defining how it will learn (optimizer), what it aims to minimize (loss function, suitable for multi-class classification where labels are one-hot encoded), and what performance indicators to monitor (accuracy).
'''

In [19]:
def unfreeze_model(model):
    # unfreeze the top 10 layers while leaving BatchNorm layers frozen for fine-tuning
    for layer in model.layers[-10:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

In [15]:
def test_model(model, test_batches):
    test_labels = test_batches.classes
    print("Test Lables", test_labels)
    print(test_batches.class_indices)

    predictions = model.predict(test_batches, step=len(test_batches), verbose=0)

    acc = 0
    for i in range(len(test_labels)):
        actual_class = test_labels[i]
        if predictions[i][actual_class] > 0.5:
            acc += 1
    print('Accuracy:', (acc/len(test_labels))*100, "%")

In [16]:
def load_data():
    train = pd.read_csv('/content/aptos2019/train_1.csv', encoding='utf-8')
    test = pd.read_csv('/content/aptos2019/test.csv', encoding='utf-8')
    valid = pd.read_csv('/content/aptos2019/valid.csv')

    train_dir = '/content/aptos2019/train_images/train_images/'
    test_dir = '/content/aptos2019/test_images/test_images/'
    valid_dir = '/content/aptos2019/val_images/val_images/'

    # construct file paths directly within function:
    train['image_path'] = train_dir + train['id_code'] + '.png'
    test['image_path'] = test_dir + test['id_code'] + '.png'
    valid['image_path'] = valid_dir + valid['id_code'] + '.png'

    train['train_images'] = train['id_code'] + '.png'
    test['test_images'] = test['id_code'] + '.png'
    valid['valid_images'] = valid['id_code'] + '.png'

    train['diagnosis'] = train['diagnosis'].astype(str)
    # train['target'] = [1 if x >= 1 else 0 for x in train['diagnosis']]
    # train['target'] = train['target'].astype(str)
    test['diagnosis'] = test['diagnosis'].astype(str)
    # test['target'] = [1 if x >= 1 else 0 for x in test['diagnosis']]
    # test['target'] = test['target'].astype(str)
    valid['diagnosis'] = valid['diagnosis'].astype(str)
    # valid['target'] = [1 if x >= 1 else 0 for x in valid['diagnosis']]
    # valid['target'] = valid['target'].astype(str)

    return train, test, valid

In [17]:
def preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis = 0)
    return preprocess_input(img_array)

In [ ]:
if __name__ == "__main__":
    train_df, test_df, valid_df = load_data()
    model = build_EfficientNetV2S_model_SVM_without_attention(NUM_CLASSES)
    unfreeze_model(model)

    train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=train_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=valid_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE)
    test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input).flow_from_dataframe(
        dataframe=test_df, x_col='image_path', y_col='diagnosis', target_size=(224,224), batch_size=BATCH_SIZE, shuffle=False)

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-4)
    history = model.fit(train_batches, epochs=epochs, validation_data=valid_batches, verbose=1,callbacks=[early_stopping,reduce_lr])

Found 2930 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.
Found 366 validated image filenames belonging to 5 classes.
Epoch 1/20
51/92 ━━━━━━━━━━━━━━━━━━━━ 2:28 4s/step - accuracy: 0.5320 - loss: 1.2371

In [ ]:
model.save('model.h5')

In [ ]:
model.save('my_model.keras')

In [ ]:
new_model = tf.keras.models.load_model('my_model.keras')

In [ ]:
new_model.summary()

In [ ]:
image = '/content/aptos2019/val_images/val_images/001639a390f0.png'

In [ ]:
img = img_to_array(load_img(image, target_size=(224, 224)))
img = np.expand_dims(img, axis=0)
img = preprocess_input(img)

In [ ]:
new_model.predict(img)

In [ ]:
s = new_model.predict(img)

In [ ]:
s

In [ ]:
s.sum()